# Stage 1: three CPI forecasts, scoring, and the disagreement test

Needs the files from `stage1_pulls` and `stage1_qa` in Drive. Run all; paste back the output of the last cell.

Locked choices (made before seeing any forecast results):
- Sample: target months Mar 2022 to Oct 2026, minus Oct 2025 and Nov 2025 (shutdown months).
- Snapshot: Kalshi quotes at 16:00 ET on the business day before the release (best quote quality of the five times checked). The nowcast is the one dated that same day, and the 2-year move is from that day's close to release-day close, so all three use the same information set.
- Price per strike: bid/ask mid; if a side is missing, the last trade.
- Tails: probability below the lowest strike sits at that strike; probability above the highest sits one step above it.
- Nowcast spread: RMSE of past day-before nowcast errors, expanding window from 2013, using only releases before the current one.

In [ ]:
MOUNT_DRIVE = True
MEASURE = "cpi"              # "cpi" (headline m/m) or "core_cpi" (core m/m); run once for each
SNAP_HOUR_ET = 16            # previous business day
# paper 3 model, fixed from the sports fit (no refitting)
LAM, G, A = 2.53, 0.55, 0.88
FEE_BASE = 0.07
# Kalshi fee multiplier for the CPI series by effective date, from your fee log. Fill in before running.
FEE_LOG = [("2021-01-01", 1.0)]
BIN_SPREAD = "sum"           # half-spread of a bin built from two thresholds: "sum" (bound on the error of a difference) or "mean"
NW_LAGS = 1                  # Newey-West lags for HAC standard errors

import numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RAW = Path("/content/drive/MyDrive/macro-news-pricing/data/raw")
else:
    RAW = Path("/content/data/raw")
ev = pd.read_parquet(RAW / "qa_events_final.parquet")
mk = pd.read_parquet(RAW / "kalshi_markets.parquet")
cd = pd.read_parquet(RAW / "kalshi_candles.parquet")
nc = pd.read_parquet(RAW / "nowcast_cpi.parquet")
act = pd.read_parquet(RAW / "nowcast_actuals.parquet")
sq = pd.read_parquet(RAW / "qa_snapshots.parquet")

## Longshot-bias correction: paper 3 model, multi-outcome version (C1, `apply_events`)
Kalshi CPI contracts are cumulative thresholds, so the ladder is first made monotone, differenced into bins covering every outcome, then corrected once with the registered one-step formula:
`n_i = m_i/S`, `pi_i = 1/(1 + ((c + lam*w(1-n_i))/w(n_i))^(1/a))` with c set so the pi sum to S, `t_i = pi_i - m_i`, `tc_i = clip(t_i, -B_i, B_i)`, `q_i = (m_i - tc_i)/S`, with `B_i = half-spread + f*m_i*(1-m_i)` and f = 0.07 x the fee multiplier in force on the snapshot date.
Two implementation choices beyond the registered formula, both reported in the output: q is floored at 0 and renormalized to sum to 1 (needed for CRPS), and a bin's half-spread comes from its two bounding thresholds (`BIN_SPREAD`).

In [ ]:
from scipy.optimize import brentq

def w(p, g=G):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return p ** g / (p ** g + (1 - p) ** g) ** (1 / g)

def fee_mult(date):
    m = FEE_LOG[0][1]
    for d, v in FEE_LOG:
        if pd.Timestamp(date) >= pd.Timestamp(d):
            m = v
    return m

def apply_events(m, half_spread, f, lam=LAM, a=A):
    """One-step multi-outcome correction (registered C1). m: bin mids, half_spread: per-bin half-spread."""
    m = np.asarray(m, float); S = m.sum()
    n = np.clip(m / S, 1e-9, 1 - 1e-9)
    wn, w1n = w(n), w(1 - n)
    pis = lambda c: 1 / (1 + ((c + lam * w1n) / wn) ** (1 / a))
    lo = -lam * w1n.min() + 1e-12
    hi = 1.0
    while pis(hi).sum() > S:
        hi *= 2
    c = brentq(lambda c: pis(c).sum() - S, lo, hi, xtol=1e-14)
    pi = pis(c)
    B = np.asarray(half_spread, float) + f * m * (1 - m)
    tc = np.clip(pi - m, -B, B)
    q = (m - tc) / S
    return q, {"c": c, "pi": pi, "B": B, "tc": tc}

# sanity check on a toy 5-bin market: longshots should be pulled down, the favourite up
_q, _ = apply_events([0.02, 0.15, 0.60, 0.20, 0.03], [0.01] * 5, f=0.07)
print("toy bins -> corrected:", np.round(_q, 4), "sum", round(_q.sum(), 4))

## Build the panel

In [ ]:
MON = {m: i for i, m in enumerate("JAN FEB MAR APR MAY JUN JUL AUG SEP OCT NOV DEC".split(), 1)}
tm = lambda t: f"20{t.split('-')[1][:2]}-{MON[t.split('-')[1][2:5]]:02d}"
DROP = {"KXCPI-25OCT", "KXCPI-25NOV", "KXCPICORE-25OCT", "KXCPICORE-25NOV"}
is_core = ev.event_ticker.str.contains("CORE")
ev = ev[is_core if MEASURE == "core_cpi" else ~is_core].copy()
ev["target_month"] = ev.event_ticker.map(tm)
ev = ev[(ev.target_month >= "2022-03") & ~ev.event_ticker.isin(DROP) & ev.print_final.notna()].copy()
# rule fixed before results: if Kalshi listed two events for one month, keep the one with more volume
dups = ev[ev.duplicated("target_month", keep=False)].event_ticker.tolist()
ev = ev.sort_values("volume").groupby("target_month").tail(1).sort_values("release_date_et")
if dups: print("duplicate months, kept highest volume of:", dups)
print(MEASURE, len(ev), "releases in sample")
ev["y"] = ev.print_final.round(1) + 0.0
ev["release_date"] = pd.to_datetime(ev.release_date_et)

cdm = cd.merge(mk[["ticker", "event_ticker", "strike"]], on="ticker", suffixes=("_x", ""))
cdm["end_et"] = cdm.end_time.dt.tz_convert("America/New_York")
mid = (cdm.yes_bid_close + cdm.yes_ask_close) / 2
both = cdm.yes_bid_close.notna() & cdm.yes_ask_close.notna()
cdm["px"] = mid.where(both, cdm.price_close)
cdm["hs"] = ((cdm.yes_ask_close - cdm.yes_bid_close) / 2).where(both)

def snapshot(r):
    t = (pd.Timestamp(r.release_date) - pd.offsets.BDay(1)).tz_localize("America/New_York") + pd.Timedelta(hours=SNAP_HOUR_ET)
    g = cdm[(cdm.event_ticker == r.event_ticker) & (cdm.end_et <= t)].dropna(subset=["px"])
    g = g.sort_values("end_et").groupby("strike").tail(1)
    return g.sort_values("strike")[["strike", "px", "hs"]], t


## Distributions on the 0.1 grid

In [ ]:
GRID = np.round(np.arange(-2.0, 3.0001, 0.1), 1)

def pav_decreasing(y):
    """Isotonic (non-increasing) fit, pool-adjacent-violators, equal weights."""
    blocks = []
    for v in y:
        blocks.append([v, 1])
        while len(blocks) > 1 and blocks[-2][0] < blocks[-1][0]:
            v2, n2 = blocks.pop(); v1, n1 = blocks.pop()
            blocks.append([(v1 * n1 + v2 * n2) / (n1 + n2), n1 + n2])
    return np.concatenate([[v] * n for v, n in blocks])

def ladder_bins(strikes, surv, hs):
    """Monotone ladder -> bins covering every outcome. Returns grid values, bin mids, bin half-spreads."""
    s = np.clip(pav_decreasing(np.asarray(surv, float)), 0, 1)
    k = np.round(np.asarray(strikes, float), 1)
    kk = np.round(np.arange(k[0], k[-1] + 0.0001, 0.1), 1)          # fill strike gaps by interpolation
    S = np.interp(kk, k, s)
    H = np.interp(kk, k, hs)
    vals = np.round(np.concatenate([[kk[0]], kk + 0.1]), 1)         # lower tail at lowest strike, then k+0.1 ...
    mids = np.concatenate([[1 - S[0]], S[:-1] - S[1:], [S[-1]]])    # ... upper tail one step above the top strike
    comb = (lambda x, y: x + y) if BIN_SPREAD == "sum" else (lambda x, y: (x + y) / 2)
    hsb = np.concatenate([[H[0]], comb(H[:-1], H[1:]), [H[-1]]])
    return vals, mids, hsb

def to_grid(vals, probs):
    pmf = pd.Series(0.0, index=GRID)
    for v, p in zip(vals, probs):
        pmf[round(v, 1)] += p
    return pmf.values

def normal_pmf(mu, sd):
    hi = stats.norm.cdf((GRID + 0.05 - mu) / sd); lo = stats.norm.cdf((GRID - 0.05 - mu) / sd)
    p = hi - lo
    return p / p.sum()

# nowcast error history: day-before nowcast vs actual, every target month since 2013
a = act[act.measure == MEASURE][["target_month", "date", "value"]].rename(columns={"date": "rel", "value": "actual"})
hist = []
for r in a.itertuples():
    d = nc[(nc.target_month == r.target_month) & (nc.date < r.rel)].dropna(subset=[MEASURE])
    if len(d):
        hist.append({"target_month": r.target_month, "rel": r.rel, "err": r.actual - d[MEASURE].iloc[-1]})
hist = pd.DataFrame(hist)

def nowcast_for(r, snap_t):
    d = nc[(nc.target_month == r.target_month) & (nc.date < r.release_date)].dropna(subset=[MEASURE])
    d = d[d.date <= snap_t.tz_localize(None)]
    past = hist[hist.rel < r.release_date].err
    return d[MEASURE].iloc[-1], np.sqrt((past ** 2).mean()), len(past)

## Forecasts and scores

In [ ]:
def scores(pmf, y, strikes):
    F = np.cumsum(pmf)
    rps = 0.1 * np.sum((F - (GRID >= y - 1e-9)) ** 2)          # CRPS on the 0.1 grid
    S = np.array([pmf[GRID > k + 1e-9].sum() for k in strikes])
    brier = np.mean((S - (y > np.asarray(strikes) + 1e-9)) ** 2)
    return rps, brier, float(GRID @ pmf)

rows = []
for r in ev.itertuples():
    lad, snap_t = snapshot(r)
    if len(lad) < 2:
        print("skip (no snapshot):", r.event_ticker); continue
    k, px = lad.strike.values, lad.px.values
    hs = lad.hs.fillna(lad.hs.median() if lad.hs.notna().any() else 0.005).values
    vals, mids, hsb = ladder_bins(k, px, hs)
    f = FEE_BASE * fee_mult(snap_t.date())
    q, info = apply_events(mids, hsb, f)
    q_raw_sum, n_neg = q.sum(), int((q < 0).sum())
    q = np.clip(q, 0, None); q = q / q.sum()
    mu, sd, nhist = nowcast_for(r, snap_t)
    pm = {"raw": to_grid(vals, mids), "corrected": to_grid(vals, q), "nowcast": normal_pmf(mu, sd)}
    row = {"event_ticker": r.event_ticker, "release_date": r.release_date, "y": r.y, "d2y_bp": r.d2y_bp,
           "n_strikes_snap": len(k), "hs_filled": int(lad.hs.isna().sum()),
           "q_sum_before_norm": q_raw_sum, "q_negative_bins": n_neg, "fee_mult": f / FEE_BASE, "nowcast_point": mu, "nowcast_sd": sd, "nowcast_hist_n": nhist}
    for name, p in pm.items():
        row[f"rps_{name}"], row[f"brier_{name}"], row[f"mean_{name}"] = scores(p, r.y, k)
    rows.append(row)
    if len(rows) == 1:
        check = pd.DataFrame({"value": vals, "m": mids, "half_spread": hsb, "pi": info["pi"], "B": info["B"], "q_corrected": q})
        check_name = r.event_ticker
res = pd.DataFrame(rows)
tight = sq[(sq.snap == "T-1 16:00") & (sq.usable_share >= 0.6)].event_ticker
res["tight_quotes"] = res.event_ticker.isin(tight)
res.to_parquet(RAW / f"stage1_results_{MEASURE}.parquet", index=False)
print(len(res), "releases scored")

## Tests

In [ ]:
def hac_mean(x):
    x = pd.Series(x).dropna()
    m = sm.OLS(x.values, np.ones(len(x))).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})
    return m.params[0], m.tvalues[0], m.pvalues[0]

def dm_table(df):
    out = []
    for a_, b_ in (("corrected", "raw"), ("corrected", "nowcast"), ("raw", "nowcast")):
        for sc in ("rps", "brier"):
            d = df[f"{sc}_{a_}"] - df[f"{sc}_{b_}"]
            m, t, p = hac_mean(d)
            wins = int((d < 0).sum()); n = int((d != 0).sum())
            out.append({"comparison": f"{a_} - {b_}", "score": sc, "mean_diff": m, "t_HAC": t, "p_HAC": p,
                        "A_better": f"{wins}/{n}", "p_sign": stats.binomtest(wins, n).pvalue if n else np.nan})
    return pd.DataFrame(out)

def reg(df, y, x):
    d = df[[y, x]].dropna()
    m = sm.OLS(d[y], sm.add_constant(d[x])).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})
    return {"y": y, "x": x, "n": len(d), "b": m.params[x], "t_HAC": m.tvalues[x], "p_HAC": m.pvalues[x], "R2": m.rsquared}

res["D"] = res.mean_corrected - res.nowcast_point          # disagreement (pp)
res["surprise"] = res.y - res.mean_corrected               # vs corrected-Kalshi mean
res["surprise_nowcast"] = res.y - res.nowcast_point
pd.set_option("display.width", 250)
for label, df in (("ALL", res), ("TIGHT QUOTES", res[res.tight_quotes])):
    print(f"\n==================== {label}: n = {len(df)}")
    print(df[[c for c in df.columns if c.startswith(("rps_", "brier_"))]].mean().round(5).to_string())
    print("\nPaired differences (negative = first forecast better):")
    print(dm_table(df).round(4).to_string(index=False))
    print("\nDisagreement regressions (HAC):")
    print(pd.DataFrame([reg(df, "surprise", "D"), reg(df, "d2y_bp", "D"),
                        reg(df, "d2y_bp", "surprise"), reg(df, "d2y_bp", "surprise_nowcast")]).round(4).to_string(index=False))
    s = df.dropna(subset=["D", "d2y_bp"]); s = s[(s.D != 0) & (s.d2y_bp != 0)]
    hits = int((np.sign(s.D) == np.sign(s.d2y_bp)).sum())
    print(f"sign(D) = sign(2y move): {hits}/{len(s)}, binomial p = {stats.binomtest(hits, len(s)).pvalue:.3f}")
print("\nnowcast sd range:", round(float(res.nowcast_sd.min()), 3), "-", round(float(res.nowcast_sd.max()), 3))
print(f"correction: fee multipliers used {sorted(res.fee_mult.unique().tolist())}, |sum q - 1| before renorm: max {abs(res.q_sum_before_norm - 1).max():.4f}, "
      f"releases with negative bins floored: {int((res.q_negative_bins > 0).sum())}, strikes with spread filled: {int(res.hs_filled.sum())}")
print(f"\nCross-check vs paper3_cpt_oos.apply_events on {check_name} (should match to rounding):")
print(check.round(5).to_string(index=False))

## Robustness
Alternative bin spread, the registered formula without rescaling, a 7:00 ET release-day snapshot, point-forecast errors, and a joint regression of the 2-year move on both surprises. Paste back this cell's output.


In [ ]:
# ---------------- Robustness: variants of the main run, point forecasts, joint surprise regression ----------------
def run_variant(bin_spread="sum", renorm=True, snap="T-1 16:00"):
    global BIN_SPREAD
    old = BIN_SPREAD; BIN_SPREAD = bin_spread
    out = []
    for r in ev.itertuples():
        if snap == "T-1 16:00":
            t = (pd.Timestamp(r.release_date) - pd.offsets.BDay(1)).tz_localize("America/New_York") + pd.Timedelta(hours=16)
        else:  # "T 07:00"
            t = pd.Timestamp(r.release_date).tz_localize("America/New_York") + pd.Timedelta(hours=7)
        g = cdm[(cdm.event_ticker == r.event_ticker) & (cdm.end_et <= t)].dropna(subset=["px"])
        lad = g.sort_values("end_et").groupby("strike").tail(1).sort_values("strike")
        if len(lad) < 2:
            continue
        k, px = lad.strike.values, lad.px.values
        hs = lad.hs.fillna(lad.hs.median() if lad.hs.notna().any() else 0.005).values
        vals, mids, hsb = ladder_bins(k, px, hs)
        q, _ = apply_events(mids, hsb, FEE_BASE * fee_mult(t.date()))
        q = np.clip(q, 1e-6, None)
        if renorm:
            q = q / q.sum()
        mu, sd, _ = nowcast_for(r, t)
        row = {"event_ticker": r.event_ticker, "y": r.y, "d2y_bp": r.d2y_bp, "nowcast_point": mu}
        for name, p in {"raw": to_grid(vals, mids), "corrected": to_grid(vals, q), "nowcast": normal_pmf(mu, sd)}.items():
            row[f"rps_{name}"], row[f"brier_{name}"], row[f"mean_{name}"] = scores(p, r.y, k)
        out.append(row)
    BIN_SPREAD = old
    return pd.DataFrame(out)

def score_line(df, label):
    d1 = df.rps_corrected - df.rps_raw; d2 = df.rps_corrected - df.rps_nowcast
    m1, t1, p1 = hac_mean(d1); m2, t2, p2 = hac_mean(d2)
    w1 = int((d1 < 0).sum()); n1 = int((d1 != 0).sum())
    return {"variant": label, "n": len(df), "rps_raw": df.rps_raw.mean(), "rps_corr": df.rps_corrected.mean(),
            "rps_nowcast": df.rps_nowcast.mean(), "corr-raw": m1, "p_HAC": p1, "corr_wins": f"{w1}/{n1}",
            "p_sign": stats.binomtest(w1, n1).pvalue if n1 else np.nan, "corr-nowcast": m2, "p_HAC_nc": p2}

variants = {
    "baseline (sum, renorm, T-1 16:00)": dict(),
    "bin spread = mean": dict(bin_spread="mean"),
    "no renorm (registered as-is)": dict(renorm=False),
    "snapshot T 07:00": dict(snap="T 07:00"),
}
runs = {lab: run_variant(**kw) for lab, kw in variants.items()}
pd.set_option("display.width", 250)
print("Score robustness (RPS; negative differences = corrected better):")
print(pd.DataFrame([score_line(df, lab) for lab, df in runs.items()]).round(4).to_string(index=False))
print("note: without renorm the corrected distribution sums to less than 1, so its RPS is not a proper score; shown because it is the registered formula")

# point forecasts: is Kalshi better on the centre, or only on the spread?
b = runs["baseline (sum, renorm, T-1 16:00)"]
err = pd.DataFrame({"raw mean": b.mean_raw - b.y, "corrected mean": b.mean_corrected - b.y, "nowcast": b.nowcast_point - b.y})
print("\nPoint forecasts vs realized print (pp):")
print(pd.DataFrame({"MAE": err.abs().mean(), "RMSE": np.sqrt((err ** 2).mean()), "bias": err.mean()}).round(4).to_string())
for a_, b_ in (("corrected mean", "nowcast"), ("corrected mean", "raw mean")):
    d = err[a_].abs() - err[b_].abs(); m, t, p = hac_mean(d)
    wins = int((d < 0).sum()); n = int((d != 0).sum())
    print(f"  |err| {a_} - {b_}: {m:+.4f} (HAC p {p:.3f}), {a_} closer in {wins}/{n}, sign p {stats.binomtest(wins, n).pvalue:.3f}")

# joint regression: which expectation does the 2-year actually react to?
b = b.assign(surprise=b.y - b.mean_corrected, surprise_nowcast=b.y - b.nowcast_point).dropna(subset=["d2y_bp"])
for xs in (["surprise"], ["surprise_nowcast"], ["surprise", "surprise_nowcast"]):
    m = sm.OLS(b.d2y_bp, sm.add_constant(b[xs])).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})
    print(f"\nd2y_bp ~ {' + '.join(xs)}  (n {int(m.nobs)}, R2 {m.rsquared:.3f})")
    print(pd.DataFrame({"b": m.params, "t_HAC": m.tvalues, "p_HAC": m.pvalues}).round(4).to_string())
m = sm.OLS(b.d2y_bp, sm.add_constant(b[["surprise", "surprise_nowcast"]])).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})
print("\nWald test, nowcast surprise adds nothing beyond Kalshi:", m.wald_test("surprise_nowcast = 0", scalar=True).pvalue.round(4))
print("Wald test, Kalshi surprise adds nothing beyond nowcast:", m.wald_test("surprise = 0", scalar=True).pvalue.round(4))


## Out-of-sample blend and trend check
Run after the Robustness cell. The blend weight is re-estimated each month from earlier releases only.


In [ ]:
# ---------------- Can a Kalshi + nowcast blend beat Kalshi out of sample? And the trend check ----------------
MIN_TRAIN = 18   # releases used before the first out-of-sample forecast (expanding window after that)

b = runs["baseline (sum, renorm, T-1 16:00)"].merge(res[["event_ticker", "release_date"]], on="event_ticker")
b = b.sort_values("release_date").reset_index(drop=True)
b["K"], b["N"] = b.mean_raw, b.nowcast_point            # market mean (raw Kalshi) and nowcast

rows = []
for i in range(MIN_TRAIN, len(b)):
    tr = b.iloc[:i]                                       # only releases before this one: no look-ahead
    x, y_ = (tr.N - tr.K).values, (tr.y - tr.K).values
    w0 = float(x @ y_ / (x @ x))                          # weight on the nowcast, no constant
    m1 = sm.OLS(y_, sm.add_constant(x)).fit()             # with a constant (bias correction too)
    r = b.iloc[i]
    rows.append({"event_ticker": r.event_ticker, "release_date": r.release_date, "y": r.y, "K": r.K, "N": r.N,
                 "w": w0, "blend": r.K + w0 * (r.N - r.K), "blend_c": r.K + m1.params[0] + m1.params[1] * (r.N - r.K)})
O = pd.DataFrame(rows)
print(f"{MEASURE}: out-of-sample releases {len(O)} ({O.release_date.min():%Y-%m} to {O.release_date.max():%Y-%m}); "
      f"nowcast weight w from {O.w.iloc[0]:.2f} (first) to {O.w.iloc[-1]:.2f} (last)")
E = pd.DataFrame({k: O[k] - O.y for k in ("K", "N", "blend", "blend_c")})
print(pd.DataFrame({"MAE": E.abs().mean(), "RMSE": np.sqrt((E ** 2).mean())}).rename(
    index={"K": "Kalshi (raw mean)", "N": "nowcast", "blend": "blend, no constant", "blend_c": "blend, with constant"}).round(4).to_string())
for k in ("blend", "blend_c"):
    for loss, f in (("abs", np.abs), ("sq", np.square)):
        d = f(E[k]) - f(E.K); m, t, p = hac_mean(d); wins = int((d < 0).sum()); n = int((d != 0).sum())
        print(f"  {k:8s} vs Kalshi, {loss} loss: {m:+.5f} (HAC p {p:.3f}), blend closer in {wins}/{n}, sign p {stats.binomtest(wins, n).pvalue:.3f}")

# trend check (core reading): is the nowcast surprise really 'deviation from the 12-month trend'?
act_m = act[act.measure == MEASURE].sort_values("target_month").set_index("target_month").value
trend = act_m.shift(1).rolling(12).mean()                 # mean of the 12 prior monthly prints (as later published)
b["target_month"] = b.event_ticker.map(tm)
b["s_k"], b["s_nc"], b["s_trend"] = b.y - b.mean_corrected, b.y - b.N, b.y - b.target_month.map(trend)
bb = b.dropna(subset=["d2y_bp", "s_trend"])
print(f"\ncorr(nowcast surprise, trend deviation) = {bb.s_nc.corr(bb.s_trend):.3f}")
for xs in (["s_k", "s_nc"], ["s_k", "s_trend"], ["s_k", "s_nc", "s_trend"]):
    m = sm.OLS(bb.d2y_bp, sm.add_constant(bb[xs])).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})
    print(f"d2y_bp ~ {' + '.join(xs)}  (n {int(m.nobs)}, R2 {m.rsquared:.3f}): " +
          ", ".join(f"{v} {m.params[v]:.1f} (p {m.pvalues[v]:.3f})" for v in xs))


## Headline and core together
Runs once both `stage1_results_cpi` and `stage1_results_core_cpi` exist (run the notebook with MEASURE = "cpi", then "core_cpi"). Which surprise does the 2-year react to?


In [ ]:
fh, fc = RAW / "stage1_results_cpi.parquet", RAW / "stage1_results_core_cpi.parquet"
if fh.exists() and fc.exists():
    h, c = pd.read_parquet(fh), pd.read_parquet(fc)
    J = h[["release_date", "d2y_bp", "y", "mean_corrected", "nowcast_point"]].merge(
        c[["release_date", "y", "mean_corrected", "nowcast_point"]], on="release_date", suffixes=("_h", "_c"))
    J["s_head"], J["s_core"] = J.y_h - J.mean_corrected_h, J.y_c - J.mean_corrected_c
    J["s_head_nc"], J["s_core_nc"] = J.y_h - J.nowcast_point_h, J.y_c - J.nowcast_point_c
    J = J.dropna(subset=["d2y_bp"])
    for xs in (["s_head"], ["s_core"], ["s_head", "s_core"], ["s_head_nc", "s_core_nc"], ["s_head", "s_core", "s_head_nc", "s_core_nc"]):
        m = sm.OLS(J.d2y_bp, sm.add_constant(J[xs])).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})
        print(f"\nd2y_bp ~ {' + '.join(xs)}  (n {int(m.nobs)}, R2 {m.rsquared:.3f})")
        print(pd.DataFrame({"b": m.params, "t_HAC": m.tvalues, "p_HAC": m.pvalues}).round(4).to_string())
    m = sm.OLS(J.d2y_bp, sm.add_constant(J[["s_head", "s_core", "s_head_nc", "s_core_nc"]])).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAGS})
    print("\nWald, nowcast surprises add nothing beyond Kalshi:", round(float(m.wald_test("s_head_nc = 0, s_core_nc = 0", scalar=True).pvalue), 4))
    print("Wald, Kalshi surprises add nothing beyond nowcast:", round(float(m.wald_test("s_head = 0, s_core = 0", scalar=True).pvalue), 4))
else:
    print("run the notebook with MEASURE = 'cpi' and with MEASURE = 'core_cpi' first")
